# 06장. Jupyter 추천 화면

| 오늘의 질문 | 예상 시간 |
|---|---:|
| 추천 기능을 누를 수 있는 화면으로 만들 수 있을까? | 4회차 후반 · 약 90분 |


## 이 장에서 배울 내용

- 입력 위젯과 출력 영역의 역할을 구분할 수 있다.
- 버튼 클릭이 추천 함수로 이어지는 흐름을 설명할 수 있다.
- 로컬 Jupyter 화면에서 가상 취향을 바꾸어 추천을 실행할 수 있다.


## 생각 열기

친구에게 추천기를 체험하게 하면서 매번 코드 속 글자를 고쳐 달라고 할 수는 없습니다. 좋아하는 메뉴를 입력하고 버튼을 누르면 결과가 나타나는 화면을 만들면, 같은 추천 함수를 더 쉽게 사용할 수 있습니다.


## 핵심 용어

| 용어 | 뜻 |
|---|---|
| **위젯** | 클릭하거나 값을 입력할 수 있는 화면 부품 |
| **콜백** | 버튼 같은 사건이 생겼을 때 실행되는 함수 |
| **상태** | 화면에 현재 들어 있는 값 |
| **프로토타입** | 핵심 기능을 시험하는 초기 완성품 |


## 개념 익히기


자동판매기의 버튼을 누르면 선택 정보가 내부 기능으로 전달되고 결과가 나옵니다. Jupyter 위젯도 같습니다.

이 판은 외부 공개 주소를 만들지 않고 현재 PC의 Jupyter 화면 안에서 작동합니다. 이름·학번 칸은 만들지 않으며 실제 알레르기·질병 정보도 입력하지 않습니다.


## 활동 전 생각


종이에 입력 상자 두 개, 선택 상자, 슬라이더, 버튼, 결과 표를 그린 뒤 각 부품에서 추천 함수의 어느 매개변수로 값이 가는지 화살표를 그어 보세요.


## 예상하기

- 좋아하는 메뉴, 기피 메뉴, 유형, 매운맛, 가상 번호 입력 부품이 보인다.
- 기본 입력 콜백은 추천표 3행을 만든다.


## 활동 1. 추천 콜백 만들기


### 코드 살펴보기


1. `run_recommendation`은 05장에서 완성한 추천 규칙을 가져옵니다.<br>
2. `recommend_from_inputs`는 화면의 글자·선택값을 추천 함수의 이름표와 연결합니다.<br>
3. `top_n=3`은 결과를 세 행으로 제한합니다.<br>
4. `preview_...` 호출은 화면을 만들기 전에 추천 기능 자체부터 확인합니다.


In [1]:
import sys
from pathlib import Path

current_folder = Path.cwd().resolve()
for candidate in (current_folder, *current_folder.parents):
    if (candidate / "jupyter_course" / "notebook_support.py").is_file():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError(
        "프로젝트 폴더를 찾지 못했습니다. 프로젝트 최상위 폴더에서 "
        r".\.venv\Scripts\python.exe -m notebook 명령으로 다시 시작하세요."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from jupyter_course.notebook_support import course_setup

setup = course_setup(PROJECT_ROOT)
PROJECT_ROOT = setup["root"]
raw_rows = setup["rows"]
meal_df = setup["frame"]
data_source = setup["source"]
print("프로젝트 폴더:", PROJECT_ROOT)
print("데이터 출처:", data_source)
print("급식 행 수:", len(raw_rows))

from neis_meal_ai.service import run_recommendation

def recommend_from_inputs(likes, avoids, menu_types, spice, fake_allergies):
    return run_recommendation(
        meal_df,
        likes_text=likes,
        avoids_text=avoids,
        preferred_types=menu_types,
        spice_level=int(spice),
        allergy_codes=[int(code) for code in fake_allergies],
        top_n=3,
    )

preview_summary, preview_table = recommend_from_inputs(
    "파스타, 피자", "오이", ["면"], 2, []
)
print(preview_summary)
print(preview_table.to_string(index=False))


프로젝트 폴더: <프로젝트 폴더>
데이터 출처: 남악고 NEIS 예비 데이터
급식 행 수: 5
취향을 비교해 3개 메뉴를 추천했습니다. 알레르기 주의 번호로 제외된 메뉴는 0개입니다.

추천 결과는 취향 비교용입니다. 실제 식단과 알레르기 정보는 학교 급식표와 영양사 안내를 다시 확인하세요.
 순위         날짜  추천 점수                                                   메뉴                                                  추천 이유      식단 군집                               알레르기 번호
  1 2026-06-24   48.7 양송이스프 미트볼로제파스타 노엣지콤비네이션피자 열대과일치즈샐러드 채소모둠피클 열무김치 아이스티 텍스트 유사도 0.15 · 좋아하는 키워드: 파스타, 피자 · 선호 유형: 면 · 매운맛 차이 1      중간 구성 1, 2, 5, 6, 9, 10, 11, 12, 13, 15, 16
  2 2026-06-29   23.4          친환경쌀밥 아욱국 족발&찐순대 배추겉절이 무말랭이무침 비빔막국수 피치에빠진코코                     텍스트 유사도 0.02 · 선호 유형: 면 · 매운맛 차이 1 상대적 든든한 구성                2, 3, 5, 6, 10, 13, 16
  3 2026-06-26   17.0         매콤떡갈비마요덮밥 감자호박된장국 구운버섯샐러드 바질크림츄볶이 배추김치 아이스구슬                                텍스트 유사도 0.00 · 매운맛 차이 1      중간 구성 1, 2, 5, 6, 9, 10, 12, 13, 15, 16, 18


### 결과 해석하기

콜백은 화면 값을 기존 추천 함수가 이해하는 형식으로 바꿉니다. 기본 호출에서 표 3행이 나오면 기능 연결이 준비된 것입니다.


## 활동 2. 입력 위젯 만들기


### 코드 살펴보기


1. `widgets.Text` 두 개는 좋아함과 피함을 쉼표로 입력받습니다.<br>
2. `SelectMultiple` 두 개는 메뉴 유형과 가상 알레르기 번호를 여러 개 선택합니다.<br>
3. `IntSlider`는 범위를 벗어난 매운맛 값이 들어오지 않게 합니다.<br>
4. Button은 사건을 만들고 Output은 안내문과 추천표가 나타날 자리를 만듭니다.


In [2]:
import os
import ipywidgets as widgets
from IPython.display import display

likes_widget = widgets.Text(value="파스타, 피자", description="좋아함")
avoids_widget = widgets.Text(value="오이", description="피함")
types_widget = widgets.SelectMultiple(
    options=["밥", "면", "국물", "튀김", "디저트"],
    value=("면",),
    description="유형",
)
spice_widget = widgets.IntSlider(value=2, min=1, max=5, description="매운맛")
allergy_widget = widgets.SelectMultiple(
    options=[str(code) for code in range(1, 20)],
    value=(),
    description="가상 번호",
)
run_button = widgets.Button(description="추천 실행", button_style="primary")
output_widget = widgets.Output()
callback_state = {"status": "not_run", "rows": 0, "message": ""}
print("입력 화면 준비 완료: 글자 입력 2개, 선택 상자 2개, 슬라이더 1개, 버튼 1개")


입력 화면 준비 완료: 글자 입력 2개, 선택 상자 2개, 슬라이더 1개, 버튼 1개


### 결과 해석하기

Text는 글자 입력, SelectMultiple은 여러 항목 선택, IntSlider는 1~5 범위 선택을 맡습니다. `callback_state`는 버튼이 실제로 성공했는지 시험하기 위한 작은 기록장입니다.


## 활동 3. 버튼 콜백 연결과 화면 조립


### 코드 살펴보기


1. `on_recommend_clicked`는 버튼을 눌렀을 때만 실행할 작업 묶음입니다.<br>
2. `try` 안에서는 입력값을 읽고, 잘못된 값이면 `except`가 한국어 수정 안내를 보여 줍니다.<br>
3. 성공하면 데이터 출처·안전 안내·추천표를 같은 출력 칸에 표시합니다.<br>
4. `on_click(...)`이 버튼과 함수를 연결하고, `VBox`가 부품을 위에서 아래로 배열합니다.<br>
5. 검증 모드에서는 콜백을 직접 한 번 실행해 행 수와 상태를 결과 계약에 기록합니다.


In [3]:
def on_recommend_clicked(_button):
    with output_widget:
        output_widget.clear_output()
        try:
            summary, table = recommend_from_inputs(
                likes_widget.value,
                avoids_widget.value,
                list(types_widget.value),
                spice_widget.value,
                list(allergy_widget.value),
            )
        except (TypeError, ValueError) as error:
            message = f"입력을 고쳐 주세요: {error}"
            callback_state.update(status="error", rows=0, message=message)
            print(message)
            return

        message = f"데이터 출처: {data_source}\n{summary}"
        callback_state.update(status="success", rows=len(table), message=message)
        print(message)
        display(table)

run_button.on_click(on_recommend_clicked)
recommender_ui = widgets.VBox([
    widgets.HTML(
        "<h3>우리 학교 급식 AI 개인추천기</h3>"
        "<p>이름·학번·실제 의료 정보는 입력하지 않습니다.</p>"
    ),
    likes_widget,
    avoids_widget,
    types_widget,
    spice_widget,
    allergy_widget,
    run_button,
    output_widget,
])

if os.getenv("NEIS_JUPYTER_VERIFY") != "1":
    display(recommender_ui)
else:
    on_recommend_clicked(None)
    print("Jupyter 위젯 콜백 실행 완료")

chapter_result = {
    "chapter": "06",
    "widget_ready": isinstance(recommender_ui, widgets.Widget),
    "callback_rows": callback_state["rows"],
    "callback_status": callback_state["status"],
    "callback_source": data_source,
}


Jupyter 위젯 콜백 실행 완료


### 결과 해석하기

화면은 로컬 Jupyter 안에 표시됩니다. 버튼은 콜백을 호출하고 콜백은 기존 추천 함수를 사용하므로 코드 셀 결과와 같은 규칙을 따릅니다.


## 탐구 활동

기본 화면의 다른 입력은 그대로 두고 `practice_spice`만 2에서 3으로 바꾸어 1위와 추천 이유를 비교하세요.

먼저 기본값으로 한 번 실행하세요. 그다음 표시된 값 하나만 바꾸고, 달라진 결과를 아래에 적습니다.


In [4]:
baseline_summary, baseline_table = recommend_from_inputs(
    "파스타, 피자", "오이", ["면"], 2, []
)
practice_spice = 3
practice_summary, practice_table = recommend_from_inputs(
    "파스타, 피자", "오이", ["면"], practice_spice, []
)
print("기준 매운맛 2의 1위:", baseline_table.iloc[0]["메뉴"])
print("매운맛만 3으로 바꾼 결과:")
print(practice_summary)
print(practice_table[["순위", "날짜", "추천 점수", "추천 이유"]].to_string(index=False))


기준 매운맛 2의 1위: 양송이스프 미트볼로제파스타 노엣지콤비네이션피자 열대과일치즈샐러드 채소모둠피클 열무김치 아이스티
매운맛만 3으로 바꾼 결과:
취향을 비교해 3개 메뉴를 추천했습니다. 알레르기 주의 번호로 제외된 메뉴는 0개입니다.

추천 결과는 취향 비교용입니다. 실제 식단과 알레르기 정보는 학교 급식표와 영양사 안내를 다시 확인하세요.
 순위         날짜  추천 점수                                                  추천 이유
  1 2026-06-24   51.7 텍스트 유사도 0.15 · 좋아하는 키워드: 파스타, 피자 · 선호 유형: 면 · 매운맛 차이 0
  2 2026-06-29   26.4                     텍스트 유사도 0.02 · 선호 유형: 면 · 매운맛 차이 0
  3 2026-06-26   20.0                                텍스트 유사도 0.00 · 매운맛 차이 0


### 내가 본 변화

- 내가 바꾼 값:  
- 화면에서 달라진 것:  
- 내 설명:


## 확인 문제

1. 버튼을 눌렀을 때 실행되는 함수를 무엇이라고 하나요?
2. 화면 코드를 추천 알고리즘과 분리하면 어떤 장점이 있나요?
3. 이 화면에 실제 알레르기 정보를 입력하면 안 되는 이유는 무엇인가요?


## 정답과 해설


1. 콜백 함수라고 합니다.<br>
2. 추천 규칙을 한 곳에서 시험하고 화면만 따로 바꿀 수 있습니다.<br>
3. 수업용 프로토타입은 의료 안전을 보장하지 않으며 민감한 개인 정보를 수집하지 않기로 했기 때문입니다.


## 핵심 정리

- 위젯은 입력과 출력을 담당하고 콜백이 기능을 연결한다.
- Jupyter판 화면은 외부 공유 링크 없이 현재 PC에서 작동한다.
- 실제 의료 정보가 아닌 가상 입력으로 동작만 시험한다.

### 다음 장

07장에서는 여러 입력 사례를 자동으로 시험하고 모델 카드로 한계를 공개합니다.


In [5]:
import json
print("__CHAPTER_RESULT__=" + json.dumps(chapter_result, ensure_ascii=False))


__CHAPTER_RESULT__={"chapter": "06", "widget_ready": true, "callback_rows": 3, "callback_status": "success", "callback_source": "남악고 NEIS 예비 데이터"}
